Day 4 — Full CNN Architecture From Scratch
Brain Tumour Detection Project
=====================================
Topics covered:
  1.  Receptive field growth with pooling — why spatial size must shrink
  2.  Channel progression — why 1→32→64→128→256
  3.  MaxPool2d internals — what it selects and why
  4.  Global Average Pooling vs Flatten — parameter count proof
  5.  Full BrainTumourCNN architecture — four ConvBlocks + GAP
  6.  Forward pass — shape at every single operation
  7.  Parameter count breakdown — every layer accounted for
  8.  Activation flow visualisation — what each block produces
  9.  Gradient flow check — healthy backprop through the full CNN
  10. Architecture summary and verification checklist
"""

In [30]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from collections import OrderedDict
from PIL import Image
import torchvision.transforms as T
 
import os
os.makedirs("outputs", exist_ok=True)
torch.manual_seed(42)

# CONVBLOCK — carried forward from Day 3
class ConvBlock(nn.Module):
    """
    Conv2d(bias=False) → BatchNorm2d → ReLU(inplace)
    Kaiming initialised. Same padding preserves spatial size.
    """
    def __init__(self, in_ch: int, out_ch: int,
                 kernel_size: int = 3, stride: int = 1):
        super().__init__()
        padding      = (kernel_size - 1) // 2
        self.conv    = nn.Conv2d(in_ch, out_ch, kernel_size,
                                 stride=stride, padding=padding, bias=False)
        self.bn      = nn.BatchNorm2d(out_ch)
        self.relu    = nn.ReLU(inplace=True)
        nn.init.kaiming_normal_(self.conv.weight,
                                mode='fan_in', nonlinearity='relu')
        nn.init.ones_(self.bn.weight)
        nn.init.zeros_(self.bn.bias)
 
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.relu(self.bn(self.conv(x)))
 

In [31]:
# 1. RECEPTIVE FIELD GROWTH WITH POOLING

"""
Without pooling (stride=1 everywhere):
  RF after L layers of 3x3 conv = 1 + 2L
  After 4 layers: RF = 9x9 = only 7% of 128px image
 
With MaxPool(2,2) between blocks:
  Pooling doubles the effective step size for all deeper layers
  RF grows much faster — covering the image in just 4 blocks
 
  Formula with pooling:
    After block 1 + pool: RF = 3,  step = 2
    After block 2 + pool: RF = 3 + 2x2 = 7,  step = 4
    After block 3 + pool: RF = 7 + 2x4 = 15,  step = 8
    After block 4:        RF = 15 + 2x8 = 31
 
  By block 4, one neuron sees a 31x31 region — 24% of image
  GAP then aggregates ALL spatial positions → sees 100%
"""



'\nWithout pooling (stride=1 everywhere):\n  RF after L layers of 3x3 conv = 1 + 2L\n  After 4 layers: RF = 9x9 = only 7% of 128px image\n\nWith MaxPool(2,2) between blocks:\n  Pooling doubles the effective step size for all deeper layers\n  RF grows much faster — covering the image in just 4 blocks\n\n  Formula with pooling:\n    After block 1 + pool: RF = 3,  step = 2\n    After block 2 + pool: RF = 3 + 2x2 = 7,  step = 4\n    After block 3 + pool: RF = 7 + 2x4 = 15,  step = 8\n    After block 4:        RF = 15 + 2x8 = 31\n\n  By block 4, one neuron sees a 31x31 region — 24% of image\n  GAP then aggregates ALL spatial positions → sees 100%\n'

In [32]:
# 2. CHANNEL PROGRESSION — WHY 32→64→128→256
"""
Each channel = one learned filter = one type of pattern detector.
 
Block 1 (32 channels):  32 different edge/gradient detectors
Block 2 (64 channels):  64 different texture detectors
Block 3 (128 channels): 128 different shape detectors
Block 4 (256 channels): 256 different high-level feature detectors
 
Why double (not triple or add 10)?
  Powers of 2 are optimal for GPU hardware (CUDA warp = 32 threads)
  Doubling gives exponential expressiveness growth
  Empirically: this 32→64→128→256 pattern works well for
  datasets of ~7000 images at 128x128
 
Why not start at 64?
  More channels in early layers = more params for simple patterns
  We want cheap early layers (few channels, full spatial size)
  and expensive late layers (many channels, small spatial size)
"""

'\nEach channel = one learned filter = one type of pattern detector.\n\nBlock 1 (32 channels):  32 different edge/gradient detectors\nBlock 2 (64 channels):  64 different texture detectors\nBlock 3 (128 channels): 128 different shape detectors\nBlock 4 (256 channels): 256 different high-level feature detectors\n\nWhy double (not triple or add 10)?\n  Powers of 2 are optimal for GPU hardware (CUDA warp = 32 threads)\n  Doubling gives exponential expressiveness growth\n  Empirically: this 32→64→128→256 pattern works well for\n  datasets of ~7000 images at 128x128\n\nWhy not start at 64?\n  More channels in early layers = more params for simple patterns\n  We want cheap early layers (few channels, full spatial size)\n  and expensive late layers (many channels, small spatial size)\n'

In [33]:
# 3. MAXPOOL2d INTERNALS
"""
MaxPool2d(kernel_size=2, stride=2):
  Divides feature map into non-overlapping 2x2 windows.
  Keeps only the MAXIMUM value in each window.
  Output size: H/2 x W/2  (exactly halves spatial dims)
 
Why max (not average)?
  A high activation = the kernel DID detect its pattern there.
  If ANY position in the 2x2 window fired strongly,
  we want to know about it — regardless of exact location.
  Average would dilute a strong detection by the zeros around it.
 
  This is also the source of local translation invariance:
  pattern detected at pixel (2,3) vs (3,3) → same max output.
  Small shifts in tumour position don't change the feature.
"""
feature_map = torch.tensor([[
    [0.1, 0.9, 0.2, 0.7],
    [0.3, 0.1, 0.8, 0.1],
    [0.6, 0.2, 0.1, 0.4],
    [0.1, 0.5, 0.3, 0.9],
]], dtype=torch.float32).unsqueeze(0)   #(1,1,4,4)

pool = nn.MaxPool2d(2, 2)
with torch.no_grad():
    out_max = pool(feature_map)
 
print("Input 4x4 feature map (single channel):")
print(feature_map[0, 0].numpy())
print("\nAfter MaxPool2d(2,2):")
print(out_max[0, 0].numpy())
print("  top-left window [0.1,0.9,0.3,0.1] → max=0.9  (strong activation preserved)")
print("  top-right window [0.2,0.7,0.8,0.1] → max=0.8  (strong activation preserved)")

H_in = 128
print(f"\nMaxPool on {H_in}x{H_in} feature map:")
print(f"  output = floor(({H_in} - 2) / 2) + 1 = {(H_in - 2) // 2 + 1}")
print(f"  = exactly {H_in // 2}x{H_in // 2}  (halves both dims)")
 

Input 4x4 feature map (single channel):
[[0.1 0.9 0.2 0.7]
 [0.3 0.1 0.8 0.1]
 [0.6 0.2 0.1 0.4]
 [0.1 0.5 0.3 0.9]]

After MaxPool2d(2,2):
[[0.9 0.8]
 [0.6 0.9]]
  top-left window [0.1,0.9,0.3,0.1] → max=0.9  (strong activation preserved)
  top-right window [0.2,0.7,0.8,0.1] → max=0.8  (strong activation preserved)

MaxPool on 128x128 feature map:
  output = floor((128 - 2) / 2) + 1 = 64
  = exactly 64x64  (halves both dims)


In [34]:
# 4. GLOBAL AVERAGE POOLING vs FLATTEN
"""
After 4 ConvBlocks + 3 MaxPools, spatial size = 8x8.
We need to connect the CNN to the MLP.
 
Option A — Flatten:
  (B, 256, 8, 8) → reshape → (B, 16384)
  First MLP layer: Linear(16384, 128)
  Parameters: 16384 x 128 = 2,097,152  (2M just for one layer!)
  Problem: spatially rigid, enormous param count, overfits
 
Option B — Global Average Pooling:
  Each 8x8 channel map → averaged to 1 scalar
  (B, 256, 8, 8) → GAP → (B, 256, 1, 1) → flatten → (B, 256)
  First MLP layer: Linear(256, 128)
  Parameters: 256 x 128 = 32,768  (65x fewer!)
  Bonus: spatial invariance — "how much" not "where exactly"
"""

x_gap_test = torch.randn(4, 256, 128, 128)
 
gap     = nn.AdaptiveAvgPool2d(output_size=1)
flatten = nn.Flatten()
 
with torch.no_grad():
    out_gap     = gap(x_gap_test)
    out_gap_flat = flatten(out_gap)
    out_flat    = x_gap_test.view(x_gap_test.size(0), -1)
 
print(f"Input tensor:              {tuple(x_gap_test.shape)}")
print(f"After GAP:                 {tuple(out_gap.shape)}")
print(f"After GAP + flatten:       {tuple(out_gap_flat.shape)}")
print(f"After raw flatten:         {tuple(out_flat.shape)}")
 
print(f"\nMLP first layer parameters:")
print(f"  With GAP:     256 x 128 = {256*128:>10,}")
print(f"  With flatten: {8*8*256} x 128 = {8*8*256*128:>10,}")
print(f"  GAP saves:    {8*8*256*128 - 256*128:>10,} parameters")

sample_channel = x_gap_test[0, 0]   # 128×128
gap_val        = sample_channel.mean().item()
print(f"\nGAP verification for one channel:")
print(f"  128x128 feature map mean = {gap_val:.6f}")
print(f"  GAP output value     = {out_gap[0, 0, 0, 0].item():.6f}")
print(f"  Match: {abs(gap_val - out_gap[0,0,0,0].item()) < 1e-6}")

Input tensor:              (4, 256, 128, 128)
After GAP:                 (4, 256, 1, 1)
After GAP + flatten:       (4, 256)
After raw flatten:         (4, 4194304)

MLP first layer parameters:
  With GAP:     256 x 128 =     32,768
  With flatten: 16384 x 128 =  2,097,152
  GAP saves:     2,064,384 parameters

GAP verification for one channel:
  128x128 feature map mean = 0.009141
  GAP output value     = 0.009141
  Match: True


In [35]:
# 5. FULL BrainTumourCNN ARCHITECTURE
class BrainTumourCNN(nn.Module):
    """
    Full CNN feature extractor for brain tumour MRI classification.
 
    Architecture:
      Block 1: ConvBlock(1→32)   + MaxPool  → 64x64
      Block 2: ConvBlock(32→64)  + MaxPool  → 32x32
      Block 3: ConvBlock(64→128) + MaxPool  → 16x16
      Block 4: ConvBlock(128→256)           →  8x8  (no pool — GAP follows)
      GAP:     AdaptiveAvgPool2d(1)         →  1x1
      Flatten: view(B, 256)                 → 256-d feature vector
 
    Input:  (B, 1, 128, 128)  — batch of grayscale MRI images
    Output: (B, 256)          — batch of 256-d feature vectors
 
    The MLP classifier head (Day 5) attaches to this output.
 
    Design choices:
      - No pool after block 4: keeps 8x8 spatial info for GAP to aggregate
      - ConvBlock order: Conv→BN→ReLU (not Conv→ReLU→BN)
        BN before ReLU normalises the full distribution before clipping
      - Kaiming init in each ConvBlock (see ConvBlock.__init__)
      - No dropout in CNN — dropout belongs in the MLP head
    """
    def __init__(self):
        super().__init__()
        self.block1 = ConvBlock(in_ch=1,   out_ch=32)
        self.block2 = ConvBlock(in_ch=32,  out_ch=64)
        self.block3 = ConvBlock(in_ch=64,  out_ch=128)
        self.block4 = ConvBlock(in_ch=128, out_ch=256)
 
        self.pool   = nn.MaxPool2d(kernel_size=2, stride=2)
        self.gap    = nn.AdaptiveAvgPool2d(output_size=1)
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        x: (B, 1, 128, 128)
        Returns: (B, 256)
        """
        x = self.pool(self.block1(x))   # (B, 32,  64, 64)
        x = self.pool(self.block2(x))   # (B, 64,  32, 32)
        x = self.pool(self.block3(x))   # (B, 128, 16, 16)
        x = self.block4(x)              # (B, 256,  8,  8)  ← no pool
        x = self.gap(x)                 # (B, 256,  1,  1)
        x = x.view(x.size(0), -1)       # (B, 256)
        return x
    def get_intermediate(self, x: torch.Tensor) -> dict:
        """
        Returns activations at every stage — used for visualisation
        and verifying shapes at each step.
        """
        activations = OrderedDict()
        activations["input"]   = x.clone()
        activations = OrderedDict()
        activations["input"]   = x.clone()
        x = self.block1(x);       activations["block1"]     = x.clone()
        x = self.pool(x);         activations["pool1"]      = x.clone()
        x = self.block2(x);       activations["block2"]     = x.clone()
        x = self.pool(x);         activations["pool2"]      = x.clone()
        x = self.block3(x);       activations["block3"]     = x.clone()
        x = self.pool(x);         activations["pool3"]      = x.clone()
        x = self.block4(x);       activations["block4"]     = x.clone()
        x = self.gap(x);          activations["gap"]        = x.clone()
        x = x.view(x.size(0),-1); activations["features"]  = x.clone()
        return activations
    

cnn = BrainTumourCNN()
cnn.train()
print(f"BrainTumourCNN created.")
print(f"  Blocks:     4 ConvBlocks")
print(f"  Pooling:    3 MaxPool2d(2,2) + 1 GlobalAvgPool")
print(f"  Input:      (B, 1, 128, 128)")
print(f"  Output:     (B, 256)")

BrainTumourCNN created.
  Blocks:     4 ConvBlocks
  Pooling:    3 MaxPool2d(2,2) + 1 GlobalAvgPool
  Input:      (B, 1, 128, 128)
  Output:     (B, 256)


In [36]:
# 6. FORWARD PASS — SHAPE AT EVERY OPERATION

x = torch.randn(4, 1, 128, 128)   # batch of 4 MRI images
cnn.train()
 
with torch.no_grad():
    activations = cnn.get_intermediate(x)
 
print(f"\n{'Stage':<15} {'Shape':>22}  {'Values per image':>18}  {'Notes'}")

notes = {
    "input":    "raw MRI pixel values",
    "block1":   "32 edge/gradient feature maps",
    "pool1":    "spatially compressed 2x",
    "block2":   "64 texture feature maps",
    "pool2":    "spatially compressed 4x",
    "block3":   "128 shape feature maps",
    "pool3":    "spatially compressed 8x",
    "block4":   "256 high-level feature maps",
    "gap":      "one value per channel",
    "features": "256-d feature vector → MLP",
}
 
for name, act in activations.items():
    shape     = tuple(act.shape)
    vals      = int(np.prod(shape[1:]))   # values per image in batch
    print(f"  {name:<13} {str(shape):>22}  {vals:>18,}  {notes[name]}")
 
print(f"\n  Total spatial compression: 128x128=16,384 → 256  ({16384/256:.0f}× reduction)")



Stage                            Shape    Values per image  Notes
  input               (4, 1, 128, 128)              16,384  raw MRI pixel values
  block1             (4, 32, 128, 128)             524,288  32 edge/gradient feature maps
  pool1                (4, 32, 64, 64)             131,072  spatially compressed 2x
  block2               (4, 64, 64, 64)             262,144  64 texture feature maps
  pool2                (4, 64, 32, 32)              65,536  spatially compressed 4x
  block3              (4, 128, 32, 32)             131,072  128 shape feature maps
  pool3               (4, 128, 16, 16)              32,768  spatially compressed 8x
  block4              (4, 256, 16, 16)              65,536  256 high-level feature maps
  gap                   (4, 256, 1, 1)                 256  one value per channel
  features                    (4, 256)                 256  256-d feature vector → MLP

  Total spatial compression: 128x128=16,384 → 256  (64× reduction)
Layer          Con

In [ ]:
# 7. PARAMETER COUNT BREAKDOWN
"""
Parameter formula per ConvBlock:
  Conv2d:     out_ch x in_ch x K x K   (no bias)
  BatchNorm:  out_ch x 2               (γ and β, one per channel)
  ReLU:       0                         (no parameters)
  Total:      out_ch x (in_ch x K x K + 2)
"""
configs = [
    ("Block 1", 1,   32,  3),
    ("Block 2", 32,  64,  3),
    ("Block 3", 64,  128, 3),
    ("Block 4", 128, 256, 3),
]
print(f"{'Layer':<12} {'Conv weights':>14} {'BN params':>12} "
      f"{'Total':>10} {'Cumulative':>12}")

cumulative = 0
for name, in_ch, out_ch, K in configs:
    conv_p = out_ch * in_ch * K * K
    bn_p   = out_ch * 2
    total  = conv_p + bn_p
    cumulative += total
    print(f"  {name:<10} {conv_p:>14,} {bn_p:>12,} {total:>10,} {cumulative:>12,}")

print(f"  {'MaxPool x3':<10} {'0':>14} {'0':>12} {'0':>10} {cumulative:>12,}")
print(f"  {'GAP':<10} {'0':>14} {'0':>12} {'0':>10} {cumulative:>12,}")

actual_total = sum(p.numel() for p in cnn.parameters())
trainable    = sum(p.numel() for p in cnn.parameters() if p.requires_grad)
print(f"  {'TOTAL':<10} {'':>14} {'':>12} {actual_total:>10,} (verified from model)")
print(f"  Trainable parameters: {trainable:,}")

print(f"  Fewer params = less overfitting risk on ~7000 MRI images")

In [42]:
# 8. ACTIVATION FLOW VISUALISATION


DATA_DIR   = "data/brain_tumour"
glioma_dir = os.path.join(DATA_DIR, "glioma")
img_file   = sorted(os.listdir(glioma_dir))[0]   # first glioma image
img_path   = os.path.join(glioma_dir, img_file)

mri_pil = Image.open(img_path).convert('L')      # force grayscale
mri_pil = mri_pil.resize((128, 128))             # match training resolution
mri     = np.array(mri_pil).astype(np.float32) / 255.0  # scale to [0,1]

print(f"Loaded: {img_path}")
print(f"Pixel range: [{mri.min():.3f}, {mri.max():.3f}]")

mri_tensor = torch.tensor(mri).unsqueeze(0).unsqueeze(0)  # (1,1,128,128)

cnn.eval()
with torch.no_grad():
    acts = cnn.get_intermediate(mri_tensor)

fig = plt.figure(figsize=(18, 10))
fig.patch.set_facecolor('#F2F2F0')
gs  = gridspec.GridSpec(5, 9, figure=fig, hspace=0.35, wspace=0.08)
 
stage_info = [
    ("input",  "Input MRI\n(1x128x128)",   None),
    ("block1", "Block 1 output\n(32x128x128)", '#378ADD'),
    ("block2", "Block 2 output\n(64x64x64)",   '#1D9E75'),
    ("block3", "Block 3 output\n(128x32x32)",  '#D85A30'),
    ("block4", "Block 4 output\n(256x8x8)",    '#9B59B6'),
]
 
for row, (stage, label, color) in enumerate(stage_info):
    act = acts[stage][0]   # remove batch dim → (C, H, W)
 
    if stage == "input":
        ax = fig.add_subplot(gs[row, 0:2])
        ax.imshow(act.squeeze().numpy(), cmap='gray')
        ax.set_title(label, fontsize=9, fontweight='bold')
        ax.axis('off')
        # stats
        ax_s = fig.add_subplot(gs[row, 2])
        ax_s.text(0.5, 0.5,
                  f"shape:\n{tuple(act.shape)}\n\n"
                  f"min: {act.min():.2f}\n"
                  f"max: {act.max():.2f}",
                  ha='center', va='center', fontsize=8,
                  transform=ax_s.transAxes)
        ax_s.axis('off')
    else:
        # Show 6 feature maps
        for fi in range(6):
            ax = fig.add_subplot(gs[row, fi])
            if fi < act.shape[0]:
                fm = act[fi].numpy()
                fm = (fm - fm.min()) / (fm.max() - fm.min() + 1e-8)
                ax.imshow(fm, cmap='viridis')
                ax.set_title(f"ch {fi}", fontsize=7, color=color)
            ax.axis('off')
 
        # Stats panel
        ax_s = fig.add_subplot(gs[row, 6:8])
        ax_s.text(0.5, 0.5,
                  f"{label}\n\n"
                  f"shape: {tuple(act.shape)}\n"
                  f"mean:  {act.mean():.3f}\n"
                  f"std:   {act.std():.3f}\n"
                  f"zeros: {(act==0).float().mean()*100:.1f}%",
                  ha='center', va='center', fontsize=8,
                  transform=ax_s.transAxes,
                  bbox=dict(boxstyle='round', facecolor='white', alpha=0.7))
        ax_s.axis('off')
 
plt.suptitle("Activation flow through BrainTumourCNN — 6 feature maps per block",
             fontsize=12, fontweight='bold', y=1.01)
plt.savefig("outputs/activation_flow.png", dpi=120, bbox_inches='tight',
            facecolor=fig.get_facecolor())

print("\nActivation sparsity per block (% of zeros after ReLU):")
for stage, label, _ in stage_info[1:]:
    act   = acts[stage][0]
    zeros = (act == 0).float().mean().item() * 100
    bar   = "░" * int(zeros / 2) + "█" * int((100-zeros)/2)
    print(f"  {stage}: {zeros:5.1f}% zeros  |{bar}|")

print("\n  ~50% zeros is healthy — ReLU kills ~half the activations")
print("  >90% zeros = dying ReLU problem")
print("  0% zeros = ReLU never firing = linear network (no non-linearity)")

Loaded: data/brain_tumour\glioma\glioma_0000.png
Pixel range: [0.000, 1.000]

Activation sparsity per block (% of zeros after ReLU):
  block1:  52.3% zeros  |░░░░░░░░░░░░░░░░░░░░░░░░░░███████████████████████|
  block2:  46.3% zeros  |░░░░░░░░░░░░░░░░░░░░░░░██████████████████████████|
  block3:  44.6% zeros  |░░░░░░░░░░░░░░░░░░░░░░███████████████████████████|
  block4:  48.0% zeros  |░░░░░░░░░░░░░░░░░░░░░░░██████████████████████████|

  ~50% zeros is healthy — ReLU kills ~half the activations
  >90% zeros = dying ReLU problem
  0% zeros = ReLU never firing = linear network (no non-linearity)


In [ ]:
# 9. GRADIENT FLOW CHECK
"""
A gradient flow check runs a dummy forward+backward pass
and inspects the gradient magnitude at each layer.
 
Healthy gradients:
  - All layers have non-zero gradients (no dead layers)
  - Gradient magnitude doesn't explode (> 1e3) or vanish (< 1e-8)
  - Earlier layers have smaller gradients than later layers
    (normal — gradient attenuates slightly going backward)
 
This check is done BEFORE real training starts.
If gradients are dead or explosive here, fix init before training.
"""

In [ ]:
"""DAY 4 SUMMARY
═════════════
Architecture:
  Input      →  (B,  1, 128, 128)
  Block1+Pool →  (B, 32,  64,  64)
  Block2+Pool →  (B, 64,  32,  32)
  Block3+Pool →  (B,128,  16,  16)
  Block4      →  (B,256,   8,   8)
  GAP         →  (B,256,   1,   1)
  Flatten     →  (B,256)           ← MLP input tomorrow
 
Total CNN parameters: {actual_params:,}
Receptive field:      31x31 after Block 4
Gradient flow:        {'healthy ✓' if all_healthy else 'check warnings above'}
 
Output files:
  outputs/activation_flow.png  ← feature maps at each block"""